# Regression Example: Used Car Price Prediction

This notebook introduces the steps to build a regression model to predict the resale price of an used car.

### Dataset

**Filename**: final_cars_maruti.csv

It is a comma separated file and there are 11 columns in the dataset.

1. Model - Model of the car
2. Location - The location in which the car was sold.
3. Age - Age of the car when the car was sold from the year of purchase.
4. KM_Driven - The total kilometers are driven in the car by the previous owner(s) in '000 kms.
5. Fuel_Type - The type of fuel used by the car. (Petrol, Diesel, Electric, CNG, LPG)
6. Transmission - The type of transmission used by the car. (Automatic / Manual)
7. Owner_Type - First, Second, Third, or Fourth & Above
8. Mileage - The standard mileage offered by the car company in kmpl or km/kg
9. Power - The maximum power of the engine in bhp.
10. Seats - The number of seats in the car.
11. Price - The resale price of the car (target).


## Loading the Dataset

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

import warnings
warnings.filterwarnings('ignore')

In [ ]:
#from google.colab import files
#uploaded = files.upload()

In [36]:
cars_df = pd.read_csv()

In [ ]:
cars_df.shape

In [ ]:
cars_df.head()

## Building a model with Seats, KM_Driven, Age, Power variables

In [ ]:
cars_df.head(2)

In [ ]:
cars_df['Model'].unique()

### Feature Set Selection

In [42]:
x_features = ['Seats', 'KM_Driven', 'Age', 'Power']

### Setting X and y variables

In [43]:
import statsmodels.api as sm

In [44]:
X = sm.add_constant(cars_df[x_features])
y = cars_df['Price']

### Data Splitting

In [45]:
from sklearn.model_selection import train_test_split

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    train_size = 0.8, 
                                                    random_state = 42)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
# OLS with y_train, X_train
lreg_v1 = sm.OLS().fit()

In [ ]:
lreg_v1.summary2()

### Multi-Collinearity

#### VIF

In [50]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def get_vif_factors( X ):
    X_matrix = X.to_numpy()
    vif = [ variance_inflation_factor( X_matrix, i ) for i in range( X_matrix.shape[1] ) ]
    vif_factors = pd.DataFrame()
    vif_factors['column'] = X.columns
    vif_factors['vif'] = vif
    
    return vif_factors

Now, calling the above method with the X features will return the VIF for the corresponding columns.

In [ ]:
vif_factors = get_vif_factors( X[list(X_train.columns)] )
vif_factors

### Residual Analysis

In [52]:
def plot_resid_fitted( fitted, resid, title):
    plt.scatter( get_standardized_values( fitted ), 
            get_standardized_values( resid ) )
    plt.title( title )
    plt.xlabel( "Standardized predicted values")
    plt.ylabel( "Standardized residual values")    
    plt.show()

In [53]:
def get_standardized_values( vals ):    
    return (vals - vals.mean())/vals.std()

In [ ]:
probplot = sm.ProbPlot( get_standardized_values( lreg_v1.resid ) );
plt.figure( figsize = (8, 6) );
probplot.ppplot( line='45' );
plt.title( "Normal P-P Plot of Regression Standardized Residuals" );
plt.show();

In [ ]:
plot_resid_fitted( lreg_v1.fittedvalues, 
                  lreg_v1.resid,
                  "Residual Plot")

### Making predictions on validation set

In [56]:
pred_y = lreg_v1.predict( X_test[X_train.columns] )

In [ ]:
from sklearn import metrics

np.sqrt(metrics.mean_squared_error(pred_y, y_test))

In [ ]:
np.round( metrics.r2_score(pred_y, y_test), 2)